# 09 Corrector

Puts the pieces together and fixes what base A gets wrong:

1. **Corrector** (CatBoost, 3 seeds): learns `y − base A`. Its inputs are the predictions of base A and base B,
   `p_artifact` and some flight context. Rows whose label is within 60 s of the delay (artefacts and on-time
   pushbacks) are left out of its training, because the mixture in step 2 adds the case that the label
   equals the delay.
2. **Mixture**: `final = corrected + W · p_artifact^α · (delay − corrected)`, the expected value over the two label
   regimes.
3. **One rule** for an extreme situation at LIRF; defined in `model/runner.py` and shown further down.
4. **LOBT window**: every 2025 label of a flight with an LOBT lies within `MVT − LOBT ± 3606 s`, so predictions are
   kept inside it.

Everything is evaluated out of fold for every month of 2025, and α and W are chosen on all twelve months together, so
they are not tuned to particular months.

In [1]:
BASE_A_RUN = "291"            # 06_base_a
BASE_B_RUN = "292"            # 07_base_b
CLASSIFIER_RUN = "294"        # 08_classifier

TRAIN_MONTHS = None           # out-of-fold months the corrector learns from (None = all available)
EVAL_MONTHS = None            # months predicted out of fold and used for the choices below (None = all available)
CORR_SEEDS = [42, 7, 123]
ALPHA_GRID = [0.75, 1.0, 1.25, 1.5]
W_GRID = [0.8, 1.0]
LOBT_WINDOW_SEC = 3606
RULES_ON = ["LIRF without NM, delay 60k-95k s"]

## Features and model

In [2]:
CATEGORICAL = ["ADEP_mvt", "STAND_mvt", "RUNWAY_mvt", "carrier", "AIRCRAFT_TYPE_mvt"]
NUMERIC = ["base_prediction", "base_normal_prediction", "p_artifact", "delay_sec", "mvt_minus_aobt_sec", "aobt_missing",
           "mvt_minus_eobt_sec", "sched_minus_eobt_sec", "aobt_minus_sched_sec", "mvt_hour", "adsb_movement_to_mvt_sec",
           "stand_reocc_gap_sec"]
FEATURES = CATEGORICAL + NUMERIC
MODEL_PARAMS = dict(iterations=1500, depth=6, learning_rate=0.05, l2_leaf_reg=10, loss_function="RMSE",
                    verbose=False, allow_writing_files=False)
ARTEFACT_TOLERANCE_SEC = 60

## Out-of-fold inputs

In [3]:
from pathlib import Path
import sys
import tempfile

import joblib
import mlflow
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "model/runner.py").is_file())
sys.path.insert(0, str(ROOT / "model"))
import runner
from runner import TARGET, TIME

runner.configure_tracking()
runs = {"A": runner.resolve_run(BASE_A_RUN), "B": runner.resolve_run(BASE_B_RUN), "C": runner.resolve_run(CLASSIFIER_RUN)}
read_oof = lambda key, columns: pd.read_parquet(runner.artifact_dir(runs[key]) / "oof_predictions.parquet", columns=columns)
oof = read_oof("A", ["row_id", "fold", "target", "prediction"]).rename(columns={"prediction": "base_prediction"})
oof = oof.merge(read_oof("B", ["row_id", "prediction"]).rename(columns={"prediction": "base_normal_prediction"}), on="row_id")
oof = oof.merge(read_oof("C", ["row_id", "p_artifact"]), on="row_id")

columns = [c for c in FEATURES if c not in ("base_prediction", "base_normal_prediction", "p_artifact")]
data = pd.read_parquet(ROOT / "data/03_features/train_DEP_merged_sorted.parquet",
                       columns=list(dict.fromkeys(columns + [TARGET, TIME, "MVT_TIME_UTC_mvt", "LOBT_flt", "MVT_ID_mvt"])))
data = data.loc[pd.to_datetime(data[TIME], utc=True).dt.year.eq(2025)].reset_index(drop=True)
assert np.allclose(oof["target"], data[TARGET].to_numpy()[oof["row_id"]]), "out-of-fold rows do not match the data"
oof = oof.join(data.drop(columns=[TARGET]), on="row_id")
oof["month"] = pd.to_datetime(oof[TIME], utc=True).dt.month
oof["lane"] = np.where(oof["aobt_missing"].eq(1), "no NM off-block", "NM off-block")
if TRAIN_MONTHS is not None:
    oof = oof.loc[oof["month"].isin(TRAIN_MONTHS)].reset_index(drop=True)
EVAL_MONTHS = sorted(oof["month"].unique()) if EVAL_MONTHS is None else EVAL_MONTHS
print(f"{len(oof):,} out-of-fold rows, evaluated months {EVAL_MONTHS}")

2,085,041 out-of-fold rows, evaluated months [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]


In [4]:
def inputs(frame):
    X = frame[FEATURES].copy()
    X[CATEGORICAL] = X[CATEGORICAL].astype("string").fillna("__MISSING__").astype(object)
    return X.astype({c: "float64" for c in NUMERIC})


def fit_corrector(frame):
    """Residual of base A; rows with label = delay (artefacts, on-time pushbacks) are left out, the mixture handles artefacts."""
    frame = frame.loc[(frame["target"] - frame["delay_sec"]).abs() > ARTEFACT_TOLERANCE_SEC]
    models = []
    for seed in CORR_SEEDS:
        model = CatBoostRegressor(**MODEL_PARAMS, random_seed=seed)
        model.fit(inputs(frame), frame["target"] - frame["base_prediction"], cat_features=CATEGORICAL)
        models.append(model)
    return models


def correct(models, frame):
    return frame["base_prediction"].to_numpy() + np.mean([m.predict(inputs(frame)) for m in models], axis=0)


def rmse_by_month(frame, column):
    """RMSE per month plus a pooled row over all evaluated months."""
    e = (frame[column] - frame["target"]) ** 2
    le = (frame["target"] <= 3600).to_numpy()
    table = pd.DataFrame({"rmse": e.groupby(frame["month"]).mean() ** 0.5,
                          "rmse_le_1h": e[le].groupby(frame.loc[le, "month"]).mean() ** 0.5})
    rows = frame["month"].isin(EVAL_MONTHS).to_numpy()
    table.loc["all"] = [e[rows].mean() ** 0.5, e[rows & le].mean() ** 0.5]
    return table

## Corrector, one month at a time

For every evaluated month the corrector is trained on all other months.

In [5]:
oof["corrected"] = np.nan
for m in EVAL_MONTHS:
    models = fit_corrector(oof.loc[oof["month"].ne(m)])
    rows = oof["month"].eq(m)
    oof.loc[rows, "corrected"] = correct(models, oof.loc[rows])
    print(f"month {m} done", flush=True)
ev = oof.loc[oof["month"].isin(EVAL_MONTHS)].copy()

month 1 done


month 2 done


month 3 done


month 4 done


month 5 done


...

month 8 done


month 9 done


month 10 done


month 11 done


month 12 done


## Mixture

α and W are chosen together on the mean RMSE over all evaluated months.

In [6]:
grid = []
for a in ALPHA_GRID:
    for w in W_GRID:
        e = (runner.mix(ev["corrected"], ev["p_artifact"], ev["delay_sec"], a, w) - ev["target"]) ** 2
        grid.append({"alpha": a, "w": w, **{f"m{m:02d}": e[ev["month"].eq(m)].mean() ** 0.5 for m in EVAL_MONTHS}})
grid = pd.DataFrame(grid)
grid["mean"] = grid[[f"m{m:02d}" for m in EVAL_MONTHS]].mean(axis=1)
ALPHA, W = grid.loc[grid["mean"].idxmin(), ["alpha", "w"]].astype(float)
ev["final"] = runner.mix(ev["corrected"], ev["p_artifact"], ev["delay_sec"], ALPHA, W)
print(f"alpha = {ALPHA}, W = {W}")
grid.sort_values("mean").head(10).round(1)

alpha = 1.0, W = 1.0


,alpha,w,m01,m02,m03,m04,m05,m06,m07,m08,m09,m10,m11,m12,mean
3,1.0,1.0,334.3,219.1,190.7,182.6,275.0,297.4,278.0,224.3,225.5,200.4,194.2,188.7,234.2
1,0.8,1.0,335.2,217.7,189.7,182.8,275.1,296.3,278.7,225.7,227.5,200.1,196.0,189.2,234.5
5,1.2,1.0,333.9,220.4,191.7,183.1,275.3,298.7,279.6,224.6,225.4,201.7,193.4,189.1,234.7
0,0.8,0.8,333.4,223.1,190.4,184.2,275.8,296.5,284.0,228.3,225.7,200.7,191.7,189.6,235.3
7,1.5,1.0,333.8,221.7,192.5,183.7,275.7,299.7,281.9,225.5,225.7,202.9,193.0,189.6,235.5
2,1.0,0.8,332.9,225.2,191.6,184.6,276.0,298.2,285.6,228.6,225.0,201.7,190.7,189.8,235.8
4,1.2,0.8,332.8,227.0,192.7,185.4,276.4,299.7,288.3,229.8,225.4,203.0,190.3,190.4,236.7
6,1.5,0.8,332.8,228.6,193.5,186.1,276.7,300.8,291.2,231.2,226.0,204.3,190.2,191.0,237.7


## Rule for an extreme case

The rule replaces the prediction by the expected value over the label types seen in 2025 for the same situation
(label = delay, = `MVT − AOBT`, = about one day, other). For the LIRF rule only two types occur, the delay and about
one day, because these flights have no AOBT. For the evaluation the shares are estimated without the month being
predicted. The table shows in how many months the rule lowers or raises the squared error.

In [7]:
train_all = pd.read_parquet(ROOT / "data/03_features/train_DEP_merged_sorted.parquet",
                            columns=["ADEP_mvt", "aobt_missing", "delay_sec", "mvt_minus_aobt_sec", TARGET, TIME])
train_all = train_all.loc[pd.to_datetime(train_all[TIME], utc=True).dt.year.eq(2025)]
train_all["month"] = pd.to_datetime(train_all[TIME], utc=True).dt.month

rule_table, before = [], ev["final"].to_numpy().copy()
for m in EVAL_MONTHS:   # shares estimated without the month being predicted
    rows = ev["month"].eq(m).to_numpy()
    estimates = {name: runner.rule_estimate(train_all.loc[runner.RULES[name](train_all) & train_all["month"].ne(m)])
                 for name in RULES_ON}
    ev.loc[rows, "final"] = runner.apply_rules(ev.loc[rows], ev.loc[rows, "final"], estimates)
gain = (before - ev["target"]) ** 2 - (ev["final"] - ev["target"]) ** 2
for name, condition in runner.RULES.items():
    rows = condition(ev)
    rule_table.append({"rule": name, "on": name in RULES_ON, "rows": int(rows.sum()),
                       "months improved": int(sum(gain[rows & ev["month"].eq(m)].sum() > 0 for m in EVAL_MONTHS)),
                       "months worse": int(sum(gain[rows & ev["month"].eq(m)].sum() < 0 for m in EVAL_MONTHS)),
                       "SSE gain all": f"{gain[rows].sum():.3g}"})
pd.DataFrame(rule_table)

,rule,on,rows,months improved,months worse,SSE gain all
0,"LIRF without NM, delay 60k-95k s",True,10,6,1,2.17e+09


## LOBT window

The taxi time of every 2025 departure with an LOBT lies within `MVT − LOBT ± 3606 s`; predictions outside are moved to
the edge.

In [8]:
ev["final"] = np.clip(runner.lobt_window(ev, ev["final"], LOBT_WINDOW_SEC), 0, None)
stages = {"base A": "base_prediction", "base B": "base_normal_prediction", "corrected": "corrected", "final": "final"}
pd.concat({stage: rmse_by_month(ev, column) for stage, column in stages.items()}, axis=1).round(1)

base A            base B            corrected             final  \
        rmse rmse_le_1h   rmse rmse_le_1h      rmse rmse_le_1h   rmse   
month                                                                   
1      340.3      199.0  458.9      180.3     338.4      185.8  328.0   
2      283.8      187.9  459.2      178.1     290.0      180.2  218.9   
3      197.6      175.5  315.1      166.3     209.6      168.9  181.5   
4      201.3      181.6  294.1      169.6     218.4      173.5  182.5   
5      284.8      181.2  324.2      172.9     290.8      174.0  275.0   
6      303.2      200.6  431.5      179.2     323.0      184.4  294.3   
7      351.7      235.7  635.7      207.7     381.0      223.2  271.0   
8      269.4      217.2  407.8      192.4     294.4      202.9  223.7   
9      238.2      207.6  439.0      182.5     247.9      188.1  229.8   
10     209.2      190.8  258.2      176.1     223.7      179.5  200.2   
11     193.0      177.2  385.3      168.6     199.3      169.7  183.0   
12     198.8      182.3  361.9      171.0     213.4      175.3  188.5   
all    262.2      196.3  409.8      179.5     276.1      185.1  236.5   

                  
      rmse_le_1h  
month             
1          186.3  
2          180.1  
3          167.3  
4          172.9  
5          170.6  
6          191.1  
7          227.7  
8          205.7  
9          193.8  
10         182.7  
11         169.1  
12         176.0  
all        186.8

In [9]:
by_lane = ((ev["final"] - ev["target"]) ** 2).groupby([ev["month"], ev["lane"]]).mean() ** 0.5
by_lane.unstack().round(1)

lane,NM off-block,no NM off-block
month,,
1,189.4,2780.4
2,195.7,1131.5
3,171.1,767.8
4,173.8,726.6
5,175.1,2283.9
6,194.4,1900.2
7,225.0,1074.5
8,200.0,928.1
9,190.6,1126.0


## Final corrector and MLflow run

Trained on all out-of-fold rows. The run stores the corrector together with α, W and the rule estimates.

In [10]:
final_models = fit_corrector(oof)
rule_estimates = {name: runner.rule_estimate(train_all.loc[runner.RULES[name](train_all)]) for name in RULES_ON}
corrector = {"models": final_models, "features": FEATURES, "categorical": CATEGORICAL, "alpha": ALPHA, "w": W,
             "rules": rule_estimates, "lobt_window_sec": LOBT_WINDOW_SEC}
experiment = mlflow.set_experiment(runner.EXPERIMENT_NAME)
with runner._start_numbered_run(experiment.experiment_id, "Corrector") as run, tempfile.TemporaryDirectory() as directory:
    directory = Path(directory)
    mlflow.log_params({"model_type": "Corrector", "base_a_run": runs["A"].info.run_id, "base_b_run": runs["B"].info.run_id,
                       "classifier_run": runs["C"].info.run_id, "alpha": ALPHA, "w": W, "eval_months": str(EVAL_MONTHS),
                       "train_months": str(TRAIN_MONTHS), "seeds": str(CORR_SEEDS), "rules": str(RULES_ON)})
    table = rmse_by_month(ev, "final")
    name = lambda m: f"m{m:02d}" if isinstance(m, (int, np.integer)) else m.replace(" ", "_")
    mlflow.log_metrics({f"rmse_{name(m)}": v for m, v in table["rmse"].items()}
                       | {f"rmse_le_1h_{name(m)}": v for m, v in table["rmse_le_1h"].items()})
    ev[["row_id", "MVT_ID_mvt", "month", "ADEP_mvt", "lane", "target", "base_prediction", "base_normal_prediction",
        "p_artifact", "corrected", "final"]].to_parquet(directory / "oof_predictions.parquet", index=False)
    joblib.dump(corrector, directory / "corrector.joblib")
    mlflow.log_artifacts(directory)
print("Run:", run.info.run_name, run.info.run_id)

Run: 295_Corrector a305f283e7b144f6aef1e085d84bf44f


## Final models

All four models are copied into `model/final` (compressed), which is versioned with the code. The prediction notebooks
(`10`, `11`) read them from there, so predictions need neither MLflow nor a new training run.

In [11]:
runs["corrector"] = run
pd.Series(runner.export_final(runs, corrector, ROOT / "model/final"), name="MB")

base_a/feature_state.joblib     0.1
base_a/median_taxi_out.py       0.0
base_a/metadata.json            0.0
base_a/model.joblib            48.8
base_b/feature_state.joblib     0.1
base_b/median_taxi_out.py       0.0
base_b/metadata.json            0.0
base_b/model.joblib            38.1
classifier.joblib              30.7
corrector.joblib               36.1
manifest.json                   0.0
Name: MB, dtype: float64